In [ ]:
%%sql -r dataframe_2
CREATE WAREHOUSE IF NOT EXISTS RETAIL_WH_P10
  WAREHOUSE_SIZE = 'XSMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = TRUE;

In [ ]:
%%sql -r dataframe_3
CREATE DATABASE IF NOT EXISTS RETAIL_DB_P10;

In [ ]:
%%sql -r dataframe_4
CREATE SCHEMA IF NOT EXISTS RETAIL_DB_P10.CUSTOMER_SCD_P10;


In [ ]:
%%sql -r dataframe_5
USE WAREHOUSE RETAIL_WH_P10;

In [ ]:
%%sql -r dataframe_6
USE DATABASE RETAIL_DB_P10;

In [ ]:
%%sql -r dataframe_7
USE SCHEMA CUSTOMER_SCD_P10;

In [ ]:
%%sql -r dataframe_9
CREATE OR REPLACE FILE FORMAT CSV_FF_P10
  TYPE = CSV
  FIELD_DELIMITER = ','
  SKIP_HEADER = 1
  FIELD_OPTIONALLY_ENCLOSED_BY = '"';

In [ ]:
%%sql -r dataframe_10
CREATE OR REPLACE STAGE CUSTOMER_STAGE_P10
  FILE_FORMAT = CSV_FF_P10;

In [ ]:
%%sql -r dataframe_1
CREATE OR REPLACE TABLE STG_CUSTOMERS_INITIAL (
  CUSTOMER_ID    NUMBER,
  CUSTOMER_NAME  VARCHAR(100),
  CITY           VARCHAR(50),
  STATE          VARCHAR(50),
  MEMBERSHIP     VARCHAR(30),
  SEGMENT        VARCHAR(30)
);

In [ ]:
%%sql -r dataframe_8
COPY INTO STG_CUSTOMERS_INITIAL
FROM @CUSTOMER_STAGE_P10/customers_initial.csv;


In [ ]:
%%sql -r dataframe_11
CREATE OR REPLACE TABLE STG_CUSTOMER_UPDATES (
  CUSTOMER_ID    NUMBER,
  CUSTOMER_NAME  VARCHAR(100),
  CITY           VARCHAR(50),
  STATE          VARCHAR(50),
  MEMBERSHIP     VARCHAR(30),
  SEGMENT        VARCHAR(30),
  EFFECTIVE_DATE DATE
);

In [ ]:
%%sql -r dataframe_12
COPY INTO STG_CUSTOMER_UPDATES
FROM @CUSTOMER_STAGE_P10/customer_updates.csv;

In [ ]:
%%sql -r dataframe_13
CREATE OR REPLACE TABLE CUSTOMER_SCD3 (
  CUSTOMER_KEY         NUMBER AUTOINCREMENT START 1 INCREMENT 1,
  CUSTOMER_ID          NUMBER,
  CUSTOMER_NAME        VARCHAR(100),
  CITY                 VARCHAR(50),
  STATE                VARCHAR(50),
  CURRENT_MEMBERSHIP   VARCHAR(30),
  PREVIOUS_MEMBERSHIP  VARCHAR(30),
  SEGMENT              VARCHAR(30)
);

In [ ]:
%%sql -r dataframe_14
INSERT INTO CUSTOMER_SCD3
  (CUSTOMER_ID, CUSTOMER_NAME, CITY, STATE, CURRENT_MEMBERSHIP, PREVIOUS_MEMBERSHIP, SEGMENT)
SELECT
  CUSTOMER_ID, CUSTOMER_NAME, CITY, STATE, MEMBERSHIP, NULL, SEGMENT
FROM STG_CUSTOMERS_INITIAL;

In [ ]:
%%sql -r dataframe_15
SELECT COUNT(*) AS TOTAL_CUSTOMERS FROM CUSTOMER_SCD3;


In [ ]:
%%sql -r dataframe_16
SELECT CUSTOMER_ID, CUSTOMER_NAME, CITY, CURRENT_MEMBERSHIP, PREVIOUS_MEMBERSHIP
FROM CUSTOMER_SCD3
ORDER BY CUSTOMER_ID;

In [ ]:
%%sql -r dataframe_17
MERGE INTO CUSTOMER_SCD3 AS tgt
USING STG_CUSTOMER_UPDATES AS src
  ON tgt.CUSTOMER_ID = src.CUSTOMER_ID
WHEN MATCHED THEN UPDATE SET
  tgt.PREVIOUS_MEMBERSHIP = tgt.CURRENT_MEMBERSHIP,
  tgt.CURRENT_MEMBERSHIP  = src.MEMBERSHIP,
  tgt.CITY                = src.CITY,
  tgt.STATE               = src.STATE,
  tgt.SEGMENT              = src.SEGMENT;

In [ ]:
%%sql -r dataframe_18
SELECT CUSTOMER_ID, CUSTOMER_NAME, CITY, CURRENT_MEMBERSHIP, PREVIOUS_MEMBERSHIP
FROM CUSTOMER_SCD3
ORDER BY CUSTOMER_ID;

In [ ]:
%%sql -r dataframe_19
SELECT CUSTOMER_ID, CUSTOMER_NAME, CURRENT_MEMBERSHIP, PREVIOUS_MEMBERSHIP
FROM CUSTOMER_SCD3
WHERE CUSTOMER_ID = 101;

In [ ]:
%%sql -r dataframe_20
CREATE OR REPLACE TABLE CUSTOMER_SCD6 (
  CUSTOMER_KEY           NUMBER AUTOINCREMENT START 1 INCREMENT 1,
  CUSTOMER_ID            NUMBER,
  CUSTOMER_NAME          VARCHAR(100),
  CITY                   VARCHAR(50),
  STATE                  VARCHAR(50),
  CURRENT_MEMBERSHIP     VARCHAR(30),
  PREVIOUS_MEMBERSHIP    VARCHAR(30),
  HISTORICAL_MEMBERSHIP  VARCHAR(30),
  SEGMENT                VARCHAR(30),
  EFFECTIVE_DATE         DATE,
  EXPIRY_DATE            DATE,
  IS_CURRENT             BOOLEAN
);

In [ ]:
%%sql -r dataframe_21
INSERT INTO CUSTOMER_SCD6
  (CUSTOMER_ID, CUSTOMER_NAME, CITY, STATE,
   CURRENT_MEMBERSHIP, PREVIOUS_MEMBERSHIP, HISTORICAL_MEMBERSHIP, SEGMENT,
   EFFECTIVE_DATE, EXPIRY_DATE, IS_CURRENT)
SELECT
  CUSTOMER_ID, CUSTOMER_NAME, CITY, STATE,
  MEMBERSHIP, NULL, MEMBERSHIP, SEGMENT,
  '2026-01-01', '9999-12-31', TRUE
FROM STG_CUSTOMERS_INITIAL;

In [ ]:
%%sql -r dataframe_22
SELECT COUNT(*) AS TOTAL_RECORDS FROM CUSTOMER_SCD6;


In [ ]:
%%sql -r dataframe_23
SELECT COUNT(*) AS CURRENT_RECORDS FROM CUSTOMER_SCD6 WHERE IS_CURRENT = TRUE;


In [ ]:
%%sql -r dataframe_24
UPDATE CUSTOMER_SCD6 tgt
SET EXPIRY_DATE = DATEADD(day, -1, src.EFFECTIVE_DATE),
    IS_CURRENT  = FALSE
FROM STG_CUSTOMER_UPDATES src
WHERE tgt.CUSTOMER_ID = src.CUSTOMER_ID
  AND tgt.IS_CURRENT  = TRUE;


In [ ]:
%%sql -r dataframe_25
INSERT INTO CUSTOMER_SCD6
  (CUSTOMER_ID, CUSTOMER_NAME, CITY, STATE,
   CURRENT_MEMBERSHIP, PREVIOUS_MEMBERSHIP, HISTORICAL_MEMBERSHIP, SEGMENT,
   EFFECTIVE_DATE, EXPIRY_DATE, IS_CURRENT)
SELECT
  src.CUSTOMER_ID, src.CUSTOMER_NAME, src.CITY, src.STATE,
  src.MEMBERSHIP,
  old.CURRENT_MEMBERSHIP,
  src.MEMBERSHIP,
  src.SEGMENT,
  src.EFFECTIVE_DATE, '9999-12-31', TRUE
FROM STG_CUSTOMER_UPDATES src
JOIN CUSTOMER_SCD6 old
  ON old.CUSTOMER_ID = src.CUSTOMER_ID
 AND old.IS_CURRENT  = FALSE
 AND old.EXPIRY_DATE = DATEADD(day, -1, src.EFFECTIVE_DATE);

In [ ]:
%%sql -r dataframe_26
SELECT CUSTOMER_ID, CUSTOMER_NAME, CURRENT_MEMBERSHIP, PREVIOUS_MEMBERSHIP,
       EFFECTIVE_DATE, EXPIRY_DATE, IS_CURRENT
FROM CUSTOMER_SCD6
ORDER BY CUSTOMER_ID, EFFECTIVE_DATE;

In [ ]:
%%sql -r dataframe_27
SELECT CUSTOMER_ID, CUSTOMER_NAME, CITY, CURRENT_MEMBERSHIP, PREVIOUS_MEMBERSHIP
FROM CUSTOMER_SCD6
WHERE IS_CURRENT = TRUE
ORDER BY CUSTOMER_ID;

In [ ]:
%%sql -r dataframe_28
SELECT CUSTOMER_ID, CUSTOMER_NAME, CURRENT_MEMBERSHIP, EFFECTIVE_DATE, EXPIRY_DATE
FROM CUSTOMER_SCD6
WHERE CUSTOMER_ID = 101
  AND '2026-03-15' BETWEEN EFFECTIVE_DATE AND EXPIRY_DATE;

In [ ]:
%%sql -r dataframe_29
SELECT COUNT(*) AS SCD_TYPE3_RECORD_COUNT FROM CUSTOMER_SCD3;


In [ ]:
%%sql -r dataframe_30
SELECT COUNT(*) AS SCD_TYPE6_RECORD_COUNT FROM CUSTOMER_SCD6;


In [ ]:
%%sql -r dataframe_31
SELECT COUNT(*) AS SCD_TYPE6_CURRENT_RECORD_COUNT
FROM CUSTOMER_SCD6 WHERE IS_CURRENT = TRUE;

In [ ]:
%%sql -r dataframe_32
SELECT COUNT(*) AS SCD_TYPE6_HISTORICAL_RECORD_COUNT
FROM CUSTOMER_SCD6 WHERE IS_CURRENT = FALSE;